In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt

In [2]:
columnas = [

    "gameid",
    "date",

    "teamname",

    "result",

    "gamelength",

    "golddiffat15",

    "xpdiffat15",

    "csdiffat15",

    "firstblood",

    "firsttower",

    "firstdragon",

    "firstherald",

    "dragons",

    "barons",

    "towers"

]

In [3]:
año = "2025"
df = pd.read_csv(f"../../Data/Raw Data/{año}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)

# Equipos

· Rendimiento:

    -Winrate
    -Duración media de partida

·Early game:

    -Media de oro al 15
    -Media de XP al 15
    -Media de CS al 15
    -Media de Kills al 15

·Objetivos:

    -Primera sangre %
    -Primera torre %
    -Primer dragón %
    -Heraldo %
    -Control de dragones%
    -Control de barones%
    -Almas %
    
·Economía:

    -Oro/minuto
    -Diferencia de oro final
    -Torres destruidas
    -Torres perdidas

In [4]:
df = df.fillna(0)
# Mantener únicamente las filas de equipos
df_teams = df[df["participantid"].isin([100, 200])]
print(df['teamname'].head(6))

# Obtengo las partidas totales (1 equipo cada fila = 2 equipos cada partida = nº filas / 2 equipos)
partidas_totales = int(df_teams["gameid"].value_counts().sum()/2)
print(partidas_totales)

# Obtengo el número de partidas jugadas de cada equipo
partidas = round(df_teams["teamname"].value_counts(),0)

print(df_teams)

0        IziDream
1        IziDream
2        IziDream
3        IziDream
4        IziDream
5    Team Valiant
Name: teamname, dtype: object
10038
                  gameid datacompleteness url league  year   split  playoffs  \
10      LOLTMNT03_179647         complete   0   LFL2  2025  Winter         0   
11      LOLTMNT03_179647         complete   0   LFL2  2025  Winter         0   
22       LOLTMNT06_96134         complete   0   LFL2  2025  Winter         0   
23       LOLTMNT06_96134         complete   0   LFL2  2025  Winter         0   
34       LOLTMNT06_95160         complete   0   LFL2  2025  Winter         0   
...                  ...              ...  ..    ...   ...     ...       ...   
120431  LOLTMNT03_332178         complete   0   DCup  2025       0         0   
120442  LOLTMNT03_331511         complete   0   DCup  2025       0         0   
120443  LOLTMNT03_331511         complete   0   DCup  2025       0         0   
120454  LOLTMNT03_332179         complete   0   DCup  20

In [5]:
df_salida = df_teams[columnas]
df_salida = df_salida.sort_values("date").reset_index(drop=True)

In [6]:
class TeamStats:

    def __init__(self):

        self.games = 0
        self.wins = 0
        self.result = 0

        #self.gamelength = 0
        self.gold15 = 0
        self.xp15 = 0
        self.cs15 = 0

        #self.firstblood = 0
        #self.firsttower = 0
        #self.firstdragon = 0
        #self.firstherald = 0

        self.dragons = 0
        self.barons = 0
        self.towers = 0


    def medias(self):

        if self.games == 0:
            return {
                "winrate": 0,
                "result": row.result,
                #"gamelength": row.gamelength,
                "gold15": row.golddiffat15,
                "xp15": row.xpdiffat15,
                "cs15": row.csdiffat15,
                #"firstblood": 0,
                #"firsttower": 0,
                #"firstdragon": 0,
                #"firstherald": 0,
                "dragons": 0,
                "barons": 0,
                "towers": 0
            }

        return {
            "winrate": self.wins / self.games,
            "result": row.result,
            #"gamelength": self.gamelength / self.games,
            "gold15": self.gold15 / self.games,
            "xp15": self.xp15 / self.games,
            "cs15": self.cs15 / self.games,
            #"firstblood": self.firstblood / self.games,
            #"firsttower": self.firsttower / self.games,
            #"firstdragon": self.firstdragon / self.games,
            #"firstherald": self.firstherald / self.games,
            "dragons": self.dragons / self.games,
            "barons": self.barons / self.games,
            "towers": self.towers / self.games
        }


    def update(self, row):

        self.games += 1

        self.wins += row.result

        self.result = row.result
        
        #self.gamelength += row.gamelength

        self.gold15 += row.golddiffat15
        self.xp15 += row.xpdiffat15
        self.cs15 += row.csdiffat15

        #self.firstblood += row.firstblood
        #self.firsttower += row.firsttower
        #self.firstdragon += row.firstdragon
        #self.firstherald += row.firstherald

        self.dragons += row.dragons
        self.barons += row.barons
        self.towers += row.towers 

In [7]:
teams = defaultdict(TeamStats)

In [8]:
dataset = []

In [9]:
for _, row in df_salida.iterrows():

    team = teams[row.teamname]

    # Solo guardamos estadísticas si ya hay partidos anteriores
    if team.games > 0:

        medias = team.medias()

        fila = {
            "gameid": row.gameid,
            "team": row.teamname,
            **medias
        }

        dataset.append(fila)

    team.update(row)

In [10]:
team_stats = pd.DataFrame(dataset)
team_stats.to_csv(

    "../../XGBoost-Model/Model_data/team_stats.csv",

    index=False

)
print(team_stats.head())

               gameid                     team  winrate  result  gold15  \
0  11715-11715_game_2                Oh My God      0.0       0     0.0   
1  11715-11715_game_2             Weibo Gaming      1.0       1     0.0   
2     LOLTMNT06_96169             Team Valiant      1.0       1  3837.0   
3     LOLTMNT06_96169  Karmine Corp Blue Stars      0.0       0   118.0   
4  11715-11715_game_3                Oh My God      0.0       0     0.0   

     xp15  cs15  dragons  barons  towers  
0     0.0   0.0      1.0     0.0     4.0  
1     0.0   0.0      4.0     1.0    10.0  
2   469.0  16.0      2.0     1.0     9.0  
3  1990.0 -43.0      0.0     0.0     3.0  
4     0.0   0.0      1.5     0.0     2.5  


# Rendimiento

## Winrate

In [11]:
# Obtengo el número de victorias de cada equipo
victorias = (
    df_teams.groupby("teamname")["result"]
      .sum()
)

In [12]:
# Función lambda para calcular el winrate en %
winrate = lambda part, vict: round((vict / part) * 100, 2)

In [13]:
team_wins = pd.DataFrame({
    "Partidas": partidas,
    "Victorias": victorias,
    "Winrate": winrate(partidas,victorias)
}).fillna(0)
print(team_wins)

                Partidas  Victorias  Winrate
teamname                                    
100 Thieves           74         41    55.41
3BL Esports            7          1    14.29
5M Esports             2          1    50.00
7REX                  10          6    60.00
A One Man Army        48         19    39.58
...                  ...        ...      ...
mCon esports          73         45    61.64
paiN Gaming           62         33    53.23
regnum4games           3          1    33.33
wangting              57         26    45.61
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4          1    25.00

[444 rows x 3 columns]


## Media de duracion de partida

In [14]:
#Obtengo las duraciones de las partidas en segundos
duraciones_s = (
    df_teams.groupby("teamname")["gamelength"]
      .sum()
)

In [15]:
# Función lambda para calcular la media de la duración de partidas
game_media = lambda duracion, part: round(duracion / part, 2)

In [16]:
team_length_media = pd.DataFrame({
    "Partidas": partidas,
    "Duración/s": duraciones_s,
    "Media": game_media(duraciones_s,partidas)
}).fillna(0)
print(team_length_media)

                Partidas  Duración/s    Media
teamname                                     
100 Thieves           74      148786  2010.62
3BL Esports            7       12406  1772.29
5M Esports             2        3874  1937.00
7REX                  10       19794  1979.40
A One Man Army        48       93111  1939.81
...                  ...         ...      ...
mCon esports          73      141277  1935.30
paiN Gaming           62      128328  2069.81
regnum4games           3        6304  2101.33
wangting              57      102656  1800.98
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4        7670  1917.50

[444 rows x 3 columns]


# **------------------------------------** 

# Early Game

## Media de oro al 15

In [17]:
# Obtengo la suma de oro al minuto 15
oro_15 = (
    df_teams.groupby("teamname")["golddiffat15"]
      .sum()
)

In [18]:
# Función lambda para calcular la media del oro al 15 del equipo
oro_media = lambda oro, part: round(oro / part, 2)

In [19]:
team_gold15_media = pd.DataFrame({
    "Partidas": partidas,
    "Oro/15": oro_15,
    "Media oro/15": oro_media(oro_15,partidas)
}).fillna(0)
print(team_gold15_media)

                Partidas   Oro/15  Media oro/15
teamname                                       
100 Thieves           74 -19750.0       -266.89
3BL Esports            7  -6684.0       -954.86
5M Esports             2  -9154.0      -4577.00
7REX                  10   8566.0        856.60
A One Man Army        48 -12839.0       -267.48
...                  ...      ...           ...
mCon esports          73  57469.0        787.25
paiN Gaming           62  -9155.0       -147.66
regnum4games           3   3232.0       1077.33
wangting              57 -29456.0       -516.77
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4  -6994.0      -1748.50

[444 rows x 3 columns]


## Media de XP al 15

In [20]:
# Obtengo la suma de xp al minuto 15
xp_15 = (
    df_teams.groupby("teamname")["xpdiffat15"]
      .sum()
)

In [21]:
# Función lambda para calcular la media de xp al 15 del equipo
xp_media = lambda xp, part: round(xp / part, 2)

In [22]:
team_xp15_media = pd.DataFrame({
    "Partidas": partidas,
    "Exp/15": xp_15,
    "Media exp/15": xp_media(xp_15,partidas)
}).fillna(0)
print(team_xp15_media)

                Partidas   Exp/15  Media exp/15
teamname                                       
100 Thieves           74 -11392.0       -153.95
3BL Esports            7  -9293.0      -1327.57
5M Esports             2  -6491.0      -3245.50
7REX                  10   6286.0        628.60
A One Man Army        48   4465.0         93.02
...                  ...      ...           ...
mCon esports          73  12645.0        173.22
paiN Gaming           62   3435.0         55.40
regnum4games           3   -221.0        -73.67
wangting              57 -40371.0       -708.26
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4  -2791.0       -697.75

[444 rows x 3 columns]


## Media de CS al 15

In [23]:
# Obtengo la suma de cs al minuto 15
cs_15 = (
    df_teams.groupby("teamname")["csdiffat15"]
      .sum()
)

In [24]:
# Función lambda para calcular la media de cs al 15 del equipo
cs_media = lambda cs, part: round(cs / part, 2)

In [25]:
team_cs15_media = pd.DataFrame({
    "Partidas": partidas,
    "Cs/15": cs_15,
    "Media cs/15": cs_media(cs_15,partidas)
}).fillna(0)
print(team_cs15_media)

                Partidas  Cs/15  Media cs/15
teamname                                    
100 Thieves           74   64.0         0.86
3BL Esports            7 -195.0       -27.86
5M Esports             2 -147.0       -73.50
7REX                  10  118.0        11.80
A One Man Army        48  647.0        13.48
...                  ...    ...          ...
mCon esports          73   78.0         1.07
paiN Gaming           62  208.0         3.35
regnum4games           3   72.0        24.00
wangting              57 -616.0       -10.81
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4  -58.0       -14.50

[444 rows x 3 columns]


## Media de Kills al 15

In [26]:
# Obtengo la suma de kills al minuto 15
kills_15 = (
    df_teams.groupby("teamname")["csat15"]
      .sum()
)

In [27]:
# Función lambda para calcular la media de kills al 15 del equipo
kills_media = lambda kills, part: round(kills / part, 2)

In [28]:
team_kills15_media = pd.DataFrame({
    "Partidas": partidas,
    "Kills/15": kills_15,
    "Media kills/15": kills_media(kills_15,partidas)
}).fillna(0)
print(team_kills15_media)

                Partidas  Kills/15  Media kills/15
teamname                                          
100 Thieves           74   39571.0          534.74
3BL Esports            7    3403.0          486.14
5M Esports             2     940.0          470.00
7REX                  10    5265.0          526.50
A One Man Army        48   24848.0          517.67
...                  ...       ...             ...
mCon esports          73   37472.0          513.32
paiN Gaming           62   33208.0          535.61
regnum4games           3    1554.0          518.00
wangting              57   27914.0          489.72
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4    2094.0          523.50

[444 rows x 3 columns]


# **------------------------------------**

# Objetivos

## % Primera sangre

In [29]:
# Obtengo el número de fb(first blood) de cada equipo
fb = (
    df_teams.groupby("teamname")["firstblood"]
      .sum()
)

In [30]:
# Función lambda para calcular el % de primeras sangre de cada equipo
fbrate = lambda part, first: round((first / part) * 100, 2)

In [31]:
team_fb = pd.DataFrame({
    "Partidas": partidas,
    "FB": fb,
    "FB_rate": fbrate(partidas,fb)
}).fillna(0)
print(team_fb)

                Partidas    FB  FB_rate
teamname                               
100 Thieves           74  35.0    47.30
3BL Esports            7   5.0    71.43
5M Esports             2   0.0     0.00
7REX                  10   6.0    60.00
A One Man Army        48  13.0    27.08
...                  ...   ...      ...
mCon esports          73  50.0    68.49
paiN Gaming           62  27.0    43.55
regnum4games           3   0.0     0.00
wangting              57  25.0    43.86
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4   2.0    50.00

[444 rows x 3 columns]


## % Primera torre

In [32]:
# Obtengo el número de ft(first tower) de cada equipo
ft = (
    df_teams.groupby("teamname")["firsttower"]
      .sum()
)

In [33]:
# Función lambda para calcular el % de primeras torres de cada equipo
ftrate = lambda part, first: round((first / part) * 100, 2)

In [34]:
team_ft = pd.DataFrame({
    "Partidas": partidas,
    "FT": ft,
    "FT_rate": ftrate(partidas,ft)
}).fillna(0)
print(team_ft)

                Partidas    FT  FT_rate
teamname                               
100 Thieves           74  34.0    45.95
3BL Esports            7   2.0    28.57
5M Esports             2   0.0     0.00
7REX                  10   8.0    80.00
A One Man Army        48  22.0    45.83
...                  ...   ...      ...
mCon esports          73  44.0    60.27
paiN Gaming           62  36.0    58.06
regnum4games           3   3.0   100.00
wangting              57  25.0    43.86
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4   2.0    50.00

[444 rows x 3 columns]


## Primer dragón en %

In [35]:
# Obtengo el número de fd(first dragon) de cada equipo
fd = (
    df_teams.groupby("teamname")["firstdragon"]
      .sum()
)
print(fd)

teamname
100 Thieves       41.0
3BL Esports        6.0
5M Esports         2.0
7REX               5.0
A One Man Army    19.0
                  ... 
mCon esports      52.0
paiN Gaming       30.0
regnum4games       1.0
wangting          25.0
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ     3.0
Name: firstdragon, Length: 444, dtype: float64


In [36]:
# Función lambda para calcular el % de primeros dragones de cada equipo
fdrate = lambda part, first: round((first / part) * 100, 2)

In [37]:
team_fd = pd.DataFrame({
    "Partidas": partidas,
    "FD": fd,
    "FD_rate": fdrate(partidas,fd)
}).fillna(0)
print(team_fd)

                Partidas    FD  FD_rate
teamname                               
100 Thieves           74  41.0    55.41
3BL Esports            7   6.0    85.71
5M Esports             2   2.0   100.00
7REX                  10   5.0    50.00
A One Man Army        48  19.0    39.58
...                  ...   ...      ...
mCon esports          73  52.0    71.23
paiN Gaming           62  30.0    48.39
regnum4games           3   1.0    33.33
wangting              57  25.0    43.86
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4   3.0    75.00

[444 rows x 3 columns]


## Heraldo en %

In [38]:
# Obtengo el número de heraldos de cada equipo
he = (
    df_teams.groupby("teamname")["heralds"]
      .sum()
)

In [39]:
# Función lambda para calcular el % de heraldos de cada equipo
herate = lambda part, first: round((first / part) * 100, 2)

In [40]:
team_he = pd.DataFrame({
    "Partidas": partidas,
    "Heraldos": he,
    "Heraldo_rate": herate(partidas,he)
}).fillna(0)
print(team_he)

                Partidas  Heraldos  Heraldo_rate
teamname                                        
100 Thieves           74      38.0         51.35
3BL Esports            7       3.0         42.86
5M Esports             2       1.0         50.00
7REX                  10       6.0         60.00
A One Man Army        48      24.0         50.00
...                  ...       ...           ...
mCon esports          73      42.0         57.53
paiN Gaming           62      25.0         40.32
regnum4games           3       2.0         66.67
wangting              57      28.0         49.12
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4       0.0          0.00

[444 rows x 3 columns]


## Control de dragones en %

In [41]:
# Obtengo el total de dragones obtenidos por los equipos
td = (
    df_teams.groupby("teamname")["dragons"]
      .sum()
)
print(td)

teamname
100 Thieves       184.0
3BL Esports        18.0
5M Esports          5.0
7REX               24.0
A One Man Army    100.0
                  ...  
mCon esports      202.0
paiN Gaming       163.0
regnum4games       10.0
wangting          117.0
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ      8.0
Name: dragons, Length: 444, dtype: float64


In [42]:
# Obtengo los dragones obtenidos y perdidos en cada partida
dragones = df_teams[['teamname', 'dragons', 'opp_dragons']].copy()
dragones = dragones.rename(columns={'dragons': 'dragones_obt'})

# Sumo para obtener el total de dragones de cada partida
dragones['dragones_tot'] = dragones['dragones_obt'] + dragones['opp_dragons']

# Elimino la columna de dragones del oponente que ya no es necesaria
dragones = dragones.drop(columns=['opp_dragons'])

# Calculo el % de control de dragones de cada partida
dragones['control_dragones'] = (dragones['dragones_obt'] / dragones['dragones_tot'] * 100).round(2)
print(dragones)

                       teamname  dragones_obt  dragones_tot  control_dragones
10                     IziDream           0.0           2.0              0.00
11                 Team Valiant           2.0           2.0            100.00
22                Esprit Shōnen           3.0           5.0             60.00
23                    Skillcamp           2.0           5.0             40.00
34      Karmine Corp Blue Stars           0.0           4.0              0.00
...                         ...           ...           ...               ...
120431               LGD Gaming           3.0           4.0             75.00
120442          Anyone's Legend           4.0           6.0             66.67
120443               LGD Gaming           2.0           6.0             33.33
120454          Anyone's Legend           0.0           4.0              0.00
120455               LGD Gaming           4.0           4.0            100.00

[20076 rows x 4 columns]


In [43]:
# Obtengo la suma del control de dragones
cd = (
    dragones.groupby("teamname")["control_dragones"]
      .sum()
)
print(cd)

teamname
100 Thieves       3922.86
3BL Esports        411.67
5M Esports         100.00
7REX               502.14
A One Man Army    2118.10
                   ...   
mCon esports      4415.14
paiN Gaming       3393.09
regnum4games       201.67
wangting          2860.01
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ     161.67
Name: control_dragones, Length: 444, dtype: float64


In [44]:
# Función lambda para calcular la media de % de control de dragones de cada equipo (como el control ya es un porcentaje solo es la división)
mcdrate = lambda control, part: round(control / part, 2)

In [45]:
control_dragones = pd.DataFrame({
    "Partidas": partidas,
    "Dragones": td,
    "D_control": mcdrate(cd,partidas)
}).fillna(0)
print(control_dragones)

                Partidas  Dragones  D_control
teamname                                     
100 Thieves           74     184.0      53.01
3BL Esports            7      18.0      58.81
5M Esports             2       5.0      50.00
7REX                  10      24.0      50.21
A One Man Army        48     100.0      44.13
...                  ...       ...        ...
mCon esports          73     202.0      60.48
paiN Gaming           62     163.0      54.73
regnum4games           3      10.0      67.22
wangting              57     117.0      50.18
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4       8.0      40.42

[444 rows x 3 columns]


## Control de barones en %

In [46]:
# Obtengo el total de barones obtenidos por los equipos
tb = (
    df_teams.groupby("teamname")["barons"]
      .sum()
)
print(tb)

teamname
100 Thieves       51.0
3BL Esports        1.0
5M Esports         1.0
7REX               8.0
A One Man Army    18.0
                  ... 
mCon esports      49.0
paiN Gaming       46.0
regnum4games       3.0
wangting          26.0
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ     1.0
Name: barons, Length: 444, dtype: float64


In [47]:
# Obtengo los barones obtenidos y perdidos en cada partida
barones = df_teams[['teamname', 'barons', 'opp_barons']].copy()
barones = barones.rename(columns={'barons': 'barones_obt'})

# Sumo para obtener el total de barones de cada partida
barones['barones_tot'] = barones['barones_obt'] + barones['opp_barons']

# Elimino la columna de barones del oponente que ya no es necesaria
barones = barones.drop(columns=['opp_barons'])

# Calculo el % de control de barones de cada partida
barones['control_barones'] = (barones['barones_obt'] / barones['barones_tot'] * 100).round(2)
print(barones)

                       teamname  barones_obt  barones_tot  control_barones
10                     IziDream          0.0          1.0              0.0
11                 Team Valiant          1.0          1.0            100.0
22                Esprit Shōnen          1.0          1.0            100.0
23                    Skillcamp          0.0          1.0              0.0
34      Karmine Corp Blue Stars          0.0          1.0              0.0
...                         ...          ...          ...              ...
120431               LGD Gaming          1.0          1.0            100.0
120442          Anyone's Legend          0.0          2.0              0.0
120443               LGD Gaming          2.0          2.0            100.0
120454          Anyone's Legend          0.0          1.0              0.0
120455               LGD Gaming          1.0          1.0            100.0

[20076 rows x 4 columns]


In [48]:
# Obtengo la suma del control de barones
cb = (
    barones.groupby("teamname")["control_barones"]
      .sum()
)
print(cb)

teamname
100 Thieves       4066.67
3BL Esports        100.00
5M Esports          50.00
7REX               500.00
A One Man Army    1400.00
                   ...   
mCon esports      3791.67
paiN Gaming       3383.33
regnum4games       200.00
wangting          1983.34
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ     100.00
Name: control_barones, Length: 444, dtype: float64


In [49]:
# Función lambda para calcular la media de % de control de barones de cada equipo (como el control ya es un porcentaje solo es la división)
mcbrate = lambda control, part: round(control / part, 2)

In [50]:
control_barones = pd.DataFrame({
    "Partidas": partidas,
    "Barones": tb,
    "B_Control": mcbrate(cb,partidas)
}).fillna(0)
print(control_barones)

                Partidas  Barones  B_Control
teamname                                    
100 Thieves           74     51.0      54.96
3BL Esports            7      1.0      14.29
5M Esports             2      1.0      25.00
7REX                  10      8.0      50.00
A One Man Army        48     18.0      29.17
...                  ...      ...        ...
mCon esports          73     49.0      51.94
paiN Gaming           62     46.0      54.57
regnum4games           3      3.0      66.67
wangting              57     26.0      34.80
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4      1.0      25.00

[444 rows x 3 columns]


## Almas en %

In [51]:
# Obtengo las partidas donde haya un alma (dragones >= 4)
n_partidas_almas = df_teams[(df_teams['dragons'] >= 4) | (df_teams['opp_dragons'] >= 4)].copy()

# Obtengo el Nº de partidas de cada equipo donde el alma haya aparecido (tanto si la obtienen como si no)
partidas_alma = n_partidas_almas['teamname'].value_counts()

# Obtengo las partidas donde hubo alma de cada equipo (donde las obtuvieron)
ts = n_partidas_almas[(n_partidas_almas['dragons'] >= 4)].copy()
ts = ts['teamname'].value_counts()

print(ts)

teamname
Los Ratones                 48
Bilibili Gaming             48
Invictus Gaming             45
Gen.G                       42
KT Rolster Challengers      42
                            ..
VeLocitY (Japanese Team)     1
BERZLOY                      1
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ               1
Inside Games                 1
Vietnam (National Team)      1
Name: count, Length: 391, dtype: int64


In [52]:
control_almas = pd.DataFrame({
    "Partidas con alma": partidas_alma,
    "Almas": ts,
    "A_control": (ts / partidas_alma * 100).round(2)
}).fillna(0)

print(control_almas)

                Partidas con alma  Almas  A_control
teamname                                           
100 Thieves                    37   20.0      54.05
3BL Esports                     2    1.0      50.00
5M Esports                      1    1.0     100.00
7REX                            5    3.0      60.00
A One Man Army                 26   11.0      42.31
...                           ...    ...        ...
mCon esports                   29   22.0      75.86
paiN Gaming                    32   18.0      56.25
regnum4games                    1    1.0     100.00
wangting                       14    9.0      64.29
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ                  3    1.0      33.33

[422 rows x 3 columns]


# ------------------------------------

# Economía

## Oro/minuto

In [53]:
# Obtengo la suma de oro por minuto
oropm = (
    df_teams.groupby("teamname")["earned gpm"]
      .sum()
)

In [54]:
# Función lambda para calcular la media del oro por minuto del equipo
oropm_media = lambda oro, part: round(oro / part, 2)

In [55]:
team_goldpm_media = pd.DataFrame({
    "Partidas": partidas,
    "Oro/15": oropm,
    "Media oro/15": oropm_media(oropm,partidas)
}).fillna(0)
print(team_goldpm_media)

                Partidas      Oro/15  Media oro/15
teamname                                          
100 Thieves           74  88859.9457       1200.81
3BL Esports            7   7630.0602       1090.01
5M Esports             2   2036.6416       1018.32
7REX                  10  12015.7067       1201.57
A One Man Army        48  55344.4007       1153.01
...                  ...         ...           ...
mCon esports          73  90772.4861       1243.46
paiN Gaming           62  73945.9208       1192.68
regnum4games           3   3655.6211       1218.54
wangting              57  71186.0863       1248.88
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4   4287.9863       1072.00

[444 rows x 3 columns]


## Diferencia de oro final

In [56]:
# Obtenemos las diferencias de oro de cada partido

# Recogemos el oro total de los equipos de cada partido en dos variables distintas (equipo 1 = variable A / equipo 2 = variable B)
oro_A = df_teams['totalgold'].iloc[0::2].values
oro_B = df_teams['totalgold'].iloc[1::2].values

# Hacemos la diferencia de para cada caso
dif_A = oro_A - oro_B  
dif_B = oro_B - oro_A  

# Inicializamos el df de diferencias con las columnas equipos y el oro inicializado
dif_oro = pd.DataFrame({
    'Equipo' : df_teams['teamname'],
    'Diferencia' : 0.0
})

# Asignamos los resultados en sus respectivas posiciones
dif_oro.iloc[0::2, dif_oro.columns.get_loc('Diferencia')] = dif_A
dif_oro.iloc[1::2, dif_oro.columns.get_loc('Diferencia')] = dif_B

print(dif_oro)

                         Equipo  Diferencia
10                     IziDream    -11681.0
11                 Team Valiant     11681.0
22                Esprit Shōnen     13990.0
23                    Skillcamp    -13990.0
34      Karmine Corp Blue Stars    -11576.0
...                         ...         ...
120431               LGD Gaming     16604.0
120442          Anyone's Legend    -12081.0
120443               LGD Gaming     12081.0
120454          Anyone's Legend    -13433.0
120455               LGD Gaming     13433.0

[20076 rows x 2 columns]


In [57]:
# Obtengo la suma de la diferencia de oro
dif_oro_total = (
    dif_oro.groupby("Equipo")["Diferencia"]
      .sum()
)

In [58]:
# Función lambda para calcular la media de las diferencias de oro
dif_oro_media = lambda dif, part: round(dif / part, 2)

In [59]:
team_gold_dif_media = pd.DataFrame({
    "Partidas": partidas,
    "Dif_o": dif_oro_total,
    "Media dif_oro": dif_oro_media(dif_oro_total,partidas)
}).fillna(0)
print(team_gold_dif_media)

                Partidas     Dif_o  Media dif_oro
100 Thieves           74  104981.0        1418.66
3BL Esports            7  -49604.0       -7086.29
5M Esports             2  -15414.0       -7707.00
7REX                  10   28554.0        2855.40
A One Man Army        48 -111840.0       -2330.00
...                  ...       ...            ...
mCon esports          73  263708.0        3612.44
paiN Gaming           62   34385.0         554.60
regnum4games           3    8425.0        2808.33
wangting              57  -46502.0        -815.82
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4  -31484.0       -7871.00

[444 rows x 3 columns]


## Torres destruidas

In [60]:
# Obtengo la suma de torretas destruidas
dtorres = (
    df_teams.groupby("teamname")["towers"]
      .sum()
)

In [61]:
# Función lambda para calcular la media de torres destruidas por el equipo
dtorres_media = lambda torres, part: round(torres / part, 2)

In [62]:
team_towers_media = pd.DataFrame({
    "Partidas": partidas,
    "Torres_d": dtorres,
    "Media torres_d": dtorres_media(dtorres,partidas)
}).fillna(0)
print(team_towers_media)

                Partidas  Torres_d  Media torres_d
teamname                                          
100 Thieves           74     462.0            6.24
3BL Esports            7      28.0            4.00
5M Esports             2       8.0            4.00
7REX                  10      64.0            6.40
A One Man Army        48     263.0            5.48
...                  ...       ...             ...
mCon esports          73     539.0            7.38
paiN Gaming           62     378.0            6.10
regnum4games           3      24.0            8.00
wangting              57     316.0            5.54
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4      15.0            3.75

[444 rows x 3 columns]


## Torres perdidas

In [63]:
# Obtengo la suma de torretas perdidas
ptorres = (
    df_teams.groupby("teamname")["opp_towers"]
      .sum()
)

In [64]:
# Función lambda para calcular la media de torres perdidas
ptorres_media = lambda torres, part: round(torres / part, 2)

In [65]:
team_towers_l_media = pd.DataFrame({
    "Partidas": partidas,
    "Torres_p": ptorres,
    "Media torres_p": ptorres_media(ptorres,partidas)
}).fillna(0)
print(team_towers_l_media)

                Partidas  Torres_p  Media torres_p
teamname                                          
100 Thieves           74     402.0            5.43
3BL Esports            7      54.0            7.71
5M Esports             2      18.0            9.00
7REX                  10      49.0            4.90
A One Man Army        48     327.0            6.81
...                  ...       ...             ...
mCon esports          73     394.0            5.40
paiN Gaming           62     355.0            5.73
regnum4games           3      17.0            5.67
wangting              57     351.0            6.16
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4      34.0            8.50

[444 rows x 3 columns]


# % victoria por lado

In [66]:
resumen_lados = df_teams.groupby(['teamname', 'side']).size()
print(resumen_lados)

teamname        side
100 Thieves     Blue    35
                Red     39
3BL Esports     Blue     3
                Red      4
5M Esports      Red      2
                        ..
regnum4games    Red      2
wangting        Blue    30
                Red     27
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ  Blue     2
                Red      2
Length: 873, dtype: int64


In [67]:
# Obtengo el número de partidas en el lado azul del equipo
blue_side = resumen_lados.unstack(fill_value=0)['Blue']
print(blue_side)
# Obtengo las partidas en el lado rojo restando al total la cantidad de partidas en el lado azul
red_side = resumen_lados.unstack(fill_value=0)['Red']
print(red_side)

teamname
100 Thieves       35
3BL Esports        3
5M Esports         0
7REX               6
A One Man Army    30
                  ..
mCon esports      36
paiN Gaming       31
regnum4games       1
wangting          30
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ     2
Name: Blue, Length: 444, dtype: int64
teamname
100 Thieves       39
3BL Esports        4
5M Esports         2
7REX               4
A One Man Army    18
                  ..
mCon esports      37
paiN Gaming       31
regnum4games       2
wangting          27
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ     2
Name: Red, Length: 444, dtype: int64


In [68]:
# Total de victorias y derrotas por lado
df_victorias_blue = df_teams[(df_teams['side'] == "Blue") & (df_teams['result'] == 1)].copy()
df_derrotas_blue = df_teams[(df_teams['side'] == "Blue") & (df_teams['result'] == 0)].copy()
df_victorias_red = df_teams[(df_teams['side'] == "Red") & (df_teams['result'] == 1)].copy()
df_derrotas_red = df_teams[(df_teams['side'] == "Red") & (df_teams['result'] == 0)].copy()

# Cantidad de victorias del equipo en el lado azul
blue_wins = df_victorias_blue['teamname'].value_counts()
blue_loose = df_derrotas_blue['teamname'].value_counts()
print(blue_wins['100 Thieves'])
print(blue_loose['100 Thieves'])

# Cantidad de victorias del equipo en el lado rojo
red_wins = df_victorias_red['teamname'].value_counts()
red_loose = df_derrotas_red['teamname'].value_counts()
print(red_wins['100 Thieves'])
print(red_loose['100 Thieves'])

17
18
24
15


In [69]:
# Función lambda para calcular el % de victoria en el lado (side) de cada equipo
swinrate = lambda part, victorias: round((victorias / part) * 100, 2)

In [70]:
team_side_winrate = pd.DataFrame({
    "Partidas": partidas,
    "Partidas lado azul": blue_wins + blue_loose,
    "Victorias lado azul": blue_wins,
    "A Winrate": swinrate(blue_wins + blue_loose, blue_wins),
    "Partidas lado rojo": red_wins + red_loose,
    "Victorias lado rojo": red_wins,
    "R Winrate": swinrate(red_wins + red_loose, blue_wins)
}).fillna(0)
print(team_side_winrate)

                Partidas  Partidas lado azul  Victorias lado azul  A Winrate  \
teamname                                                                       
100 Thieves           74                35.0                 17.0      48.57   
3BL Esports            7                 0.0                  0.0       0.00   
5M Esports             2                 0.0                  0.0       0.00   
7REX                  10                 6.0                  3.0      50.00   
A One Man Army        48                30.0                 16.0      53.33   
...                  ...                 ...                  ...        ...   
mCon esports          73                36.0                 27.0      75.00   
paiN Gaming           62                31.0                 17.0      54.84   
regnum4games           3                 0.0                  0.0       0.00   
wangting              57                30.0                 16.0      53.33   
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         4                

# Jugadores

·Rendimiento

    -KDA
    -Participation de Kill
    -Daño por minuto
    -Oro por minuto
    -CS/min
    -Oro/min
    
·Early game

    -Diferencia de oro al 15
    -Diferencia de XP al 15
    -Diferencia de CS al 15
    
·Visión

    -Vision/min
    -Wards colocados/min
    -Wards destruidos/min
    

In [71]:
# Mantener únicamente las filas de jugadores
df_players = df[df["participantid"].between(1, 10)]

partidas = round(df_players["playername"].value_counts(),0)
print(partidas)

playername
Shanks         238
Kanavi         200
Elk            190
ON             189
Bin            189
              ... 
Chova            1
Hildebrandt      1
ReigN            1
GekkoSzaby       1
PAPU its me      1
Name: count, Length: 2803, dtype: int64


# Rendimiento

## KDA

In [72]:
# Obtenemos el KDA del jugador de cada partido

# Creamos el nuevo df que guardará los datos a operar
kda = pd.DataFrame({
    'Jugador' : df_players['playername'],
    'kills' : df_players['kills'],
    'assists' : df_players['assists'],
    'deaths' : df_players['deaths']
})

#kda['KDA'] = round((df_players['kills'] + df_players['assists']) / np.where(df_players['deaths'] >=1, df_players['deaths'], 1),2)
print(kda)

          Jugador  kills  assists  deaths
0        PatkicaA      1        1       2
1          Joinze      0        1       3
2            Sayn      1        0       2
3       Shiganari      1        1       3
4          Lekcyc      0        2       3
...           ...    ...      ...     ...
120449       sasi      3        6       2
120450       Heng      4       12       2
120451   Tangyuan     14        7       2
120452     Shaoye      0       12       3
120453        Ycx      1       18       1

[100380 rows x 4 columns]


In [73]:
# Obtengo la suma de los KDA
kills_total = (
    kda.groupby("Jugador")["kills"]
      .sum()
)
assists_total = (
    kda.groupby("Jugador")["assists"]
      .sum()
)
deaths_total = (
    kda.groupby("Jugador")["deaths"]
      .sum()
)
kda_total = round((kills_total + assists_total) / np.where(deaths_total >=1, deaths_total, 1),2)
print(kda_total)

Jugador
 BLWeaksideWizar    1.00
0909                3.00
0ri                 2.30
1116                3.76
113                 2.93
                    ... 
zeta                3.86
zorenous            1.43
zynts               3.36
zzeta               1.21
Évangelyne          6.59
Length: 2803, dtype: float64


In [74]:
# Función lambda para calcular la media del KDA
kda_media = lambda tot, part: round(tot / part, 2)

In [75]:
player_kda_media = pd.DataFrame({
    "Partidas": partidas,
    "Sum KDA": kda_total,
    "Media KDA": kda_media(kda_total,partidas)
}).fillna(0)
print(player_kda_media)

                  Partidas  Sum KDA  Media KDA
 BLWeaksideWizar         2     1.00       0.50
0909                     3     3.00       1.00
0ri                     29     2.30       0.08
1116                    74     3.76       0.05
113                     67     2.93       0.04
...                    ...      ...        ...
zeta                     5     3.86       0.77
zorenous                 4     1.43       0.36
zynts                   47     3.36       0.07
zzeta                    5     1.21       0.24
Évangelyne              21     6.59       0.31

[2803 rows x 3 columns]


## Participación de kill

In [76]:
# Obtenemos el KDA del jugador de cada partido

# Creamos el nuevo df que guardará los datos a operar
kpart = pd.DataFrame({
    'Jugador' : df_players['playername'],
    'kills' : df_players['kills'],
    'assists' : df_players['assists'],
    'teamkills' : df_players['teamkills']
})

#kpart['Participación de kills'] = round((df_players['kills'] + df_players['assists']) / np.where(df_players['teamkills'] >=1, df_players['teamkills'], 1) * 100,2)
print(kpart)

          Jugador  kills  assists  teamkills
0        PatkicaA      1        1          3
1          Joinze      0        1          3
2            Sayn      1        0          3
3       Shiganari      1        1          3
4          Lekcyc      0        2          3
...           ...    ...      ...        ...
120449       sasi      3        6         22
120450       Heng      4       12         22
120451   Tangyuan     14        7         22
120452     Shaoye      0       12         22
120453        Ycx      1       18         22

[100380 rows x 4 columns]


In [77]:
# Obtengo la suma de los % de participación
kills_total = (
    kpart.groupby("Jugador")['kills']
      .sum()
)
assists_total = (
    kpart.groupby("Jugador")['assists']
      .sum()
)
teamkills_total = (
    kpart.groupby("Jugador")['teamkills']
      .sum()
)

kpart_total = round((kills_total + assists_total) / np.where(teamkills_total >=1, teamkills_total, 1) * 100,2)

print(kpart_total)

Jugador
 BLWeaksideWizar    38.10
0909                81.08
0ri                 71.36
1116                65.73
113                 76.80
                    ...  
zeta                80.19
zorenous            50.00
zynts               53.11
zzeta               58.00
Évangelyne          59.37
Length: 2803, dtype: float64


In [78]:
player_kpart_media = pd.DataFrame({
    "Partidas": partidas,
    "Part_kills": kpart_total,
}).fillna(0)
print(player_kpart_media)

                  Partidas  Part_kills
 BLWeaksideWizar         2       38.10
0909                     3       81.08
0ri                     29       71.36
1116                    74       65.73
113                     67       76.80
...                    ...         ...
zeta                     5       80.19
zorenous                 4       50.00
zynts                   47       53.11
zzeta                    5       58.00
Évangelyne              21       59.37

[2803 rows x 2 columns]


## Daño por minuto

In [79]:
# Obtengo la suma de daño por minuto
dpm = (
    df_players.groupby("playername")["dpm"]
      .sum()
)

In [80]:
# Función lambda para calcular la media del daño por minuto del jugador
dmgpm_media = lambda dmg, part: round(dmg / part, 2)

In [81]:
player_dpm_media = pd.DataFrame({
    "Partidas": partidas,
    "Daño/m": dpm,
    "Media daño/m": dmgpm_media(dpm,partidas)
}).fillna(0)
print(player_dpm_media)

                  Partidas      Daño/m  Media daño/m
playername                                          
 BLWeaksideWizar         2   1200.4080        600.20
0909                     3   2425.8159        808.61
0ri                     29  12742.9980        439.41
1116                    74  63789.7358        862.02
113                     67  33655.4589        502.32
...                    ...         ...           ...
zeta                     5   1204.4818        240.90
zorenous                 4   2334.6554        583.66
zynts                   47  34073.3117        724.96
zzeta                    5   3010.8813        602.18
Évangelyne              21  13374.0413        636.86

[2803 rows x 3 columns]


## Oro por minuto

In [82]:
# Obtengo la suma de daño por minuto
gpm = (
    df_players.groupby("playername")["earned gpm"]
      .sum()
)

In [83]:
# Función lambda para calcular la media del daño por minuto del jugador
gpm_media = lambda oro, part: round(oro / part, 2)

In [84]:
player_gpm_media = pd.DataFrame({
    "Partidas": partidas,
    "Oro/m": gpm,
    "Media oro/m": gpm_media(gpm,partidas)
}).fillna(0)
print(player_gpm_media)

                  Partidas       Oro/m  Media oro/m
playername                                         
 BLWeaksideWizar         2    406.6844       203.34
0909                     3    871.2767       290.43
0ri                     29   6476.3482       223.32
1116                    74  22976.2238       310.49
113                     67  14491.7887       216.30
...                    ...         ...          ...
zeta                     5    649.9552       129.99
zorenous                 4    898.0439       224.51
zynts                   47  13402.2870       285.16
zzeta                    5   1410.1898       282.04
Évangelyne              21   6527.9124       310.85

[2803 rows x 3 columns]


## cs/min

In [85]:
# Obtengo la suma de cs por minuto
cspm = (
    df_teams.groupby("teamname")["cspm"]
      .sum()
)

In [86]:
# Función lambda para calcular la media de cs por minuto del jugador
cspm_media = lambda cs, part: round(cs / part, 2)

In [87]:
player_cspm_media = pd.DataFrame({
    "Partidas": partidas,
    "CS/m": cspm,
    "Media cs/m": cspm_media(cspm,partidas)
}).fillna(0)
print(player_cspm_media)

                  Partidas       CS/m  Media cs/m
 BLWeaksideWizar       2.0     0.0000         0.0
0909                   3.0     0.0000         0.0
0ri                   29.0     0.0000         0.0
100 Thieves            0.0  2434.9496         0.0
1116                  74.0     0.0000         0.0
...                    ...        ...         ...
zorenous               4.0     0.0000         0.0
zynts                 47.0     0.0000         0.0
zzeta                  5.0     0.0000         0.0
Évangelyne            21.0     0.0000         0.0
ΣΤΕΝΑΧΩΡΕΜΕΝΟΙ         0.0   125.6685         0.0

[3247 rows x 3 columns]


# ------------------------------------

# Early game

## Diferencia de oro al 15

In [88]:
# Obtengo la suma de oro al minuto 15
poro_15 = (
    df_players.groupby("playername")["golddiffat15"]
      .sum()
)

In [89]:
# Función lambda para calcular la media del oro al 15 del jugador
poro_media = lambda oro, part: round(oro / part, 2)

In [90]:
player_gold15_media = pd.DataFrame({
    "Partidas": partidas,
    "Dif/15": poro_15,
    "Media dif/15": poro_media(poro_15,partidas)
}).fillna(0)
print(player_gold15_media)

                  Partidas   Dif/15  Media dif/15
playername                                       
 BLWeaksideWizar         2  -2358.0      -1179.00
0909                     3  -2338.0       -779.33
0ri                     29   -782.0        -26.97
1116                    74  39980.0        540.27
113                     67  11550.0        172.39
...                    ...      ...           ...
zeta                     5    497.0         99.40
zorenous                 4    -60.0        -15.00
zynts                   47   5726.0        121.83
zzeta                    5  -1904.0       -380.80
Évangelyne              21   -299.0        -14.24

[2803 rows x 3 columns]


## Diferencia de XP al 15

In [91]:
# Obtengo la suma de xp al minuto 15
pxp_15 = (
    df_players.groupby("playername")["xpdiffat15"]
      .sum()
)

In [92]:
# Función lambda para calcular la media de xp al 15 del jugador
pxp_media = lambda xp, part: round(xp / part, 2)

In [93]:
player_xp15_media = pd.DataFrame({
    "Partidas": partidas,
    "Dif/15": pxp_15,
    "Media dif/15": pxp_media(pxp_15,partidas)
}).fillna(0)
print(player_xp15_media)

                  Partidas   Dif/15  Media dif/15
playername                                       
 BLWeaksideWizar         2   -425.0       -212.50
0909                     3  -1920.0       -640.00
0ri                     29  -4772.0       -164.55
1116                    74  12085.0        163.31
113                     67  -3358.0        -50.12
...                    ...      ...           ...
zeta                     5   -354.0        -70.80
zorenous                 4   -842.0       -210.50
zynts                   47   2756.0         58.64
zzeta                    5  -2895.0       -579.00
Évangelyne              21   3695.0        175.95

[2803 rows x 3 columns]


## Diferencia de CS al 15

In [94]:
# Obtengo la suma de cs al minuto 15
pcs_15 = (
    df_players.groupby("playername")["csdiffat15"]
      .sum()
)

In [95]:
# Función lambda para calcular la media de cs al 15 del jugador
pcs_media = lambda cs, part: round(cs / part, 2)

In [96]:
player_cs15_media = pd.DataFrame({
    "Partidas": partidas,
    "Dif/15": pcs_15,
    "Media dif/15": pcs_media(pcs_15,partidas)
}).fillna(0)
print(player_cs15_media)

                  Partidas  Dif/15  Media dif/15
playername                                      
 BLWeaksideWizar         2    -4.0         -2.00
0909                     3    -4.0         -1.33
0ri                     29   -52.0         -1.79
1116                    74   669.0          9.04
113                     67   -92.0         -1.37
...                    ...     ...           ...
zeta                     5    10.0          2.00
zorenous                 4    10.0          2.50
zynts                   47   245.0          5.21
zzeta                    5   -63.0        -12.60
Évangelyne              21    50.0          2.38

[2803 rows x 3 columns]


# ------------------------------------

# Visión

## Vision/min

In [97]:
# Obtengo la suma de visión por minuto
vpm = (
    df_players.groupby("playername")["vspm"]
      .sum()
)

In [98]:
# Función lambda para calcular la media de visión por minuto del jugador
vpm_media = lambda vision, part: round(vision / part, 2)

In [99]:
player_vpm_media = pd.DataFrame({
    "Partidas": partidas,
    "Vis/m": vpm,
    "Media vis/m": vpm_media(vpm,partidas)
}).fillna(0)
print(player_vpm_media)

                  Partidas    Vis/m  Media vis/m
playername                                      
 BLWeaksideWizar         2   1.3525         0.68
0909                     3   2.2448         0.75
0ri                     29  25.6750         0.89
1116                    74  67.5695         0.91
113                     67  80.8514         1.21
...                    ...      ...          ...
zeta                     5  12.9315         2.59
zorenous                 4   4.0571         1.01
zynts                   47  55.7686         1.19
zzeta                    5   4.6183         0.92
Évangelyne              21  16.8287         0.80

[2803 rows x 3 columns]


## Wards colocados/min

In [100]:
# Obtengo la suma de guardianes colocados por minuto
wpm = (
    df_players.groupby("playername")["wpm"]
      .sum()
)

In [101]:
# Función lambda para calcular la media de guardianes colocados por minuto del jugador
wpm_media = lambda wards, part: round(wards / part, 2)

In [102]:
player_wpm_media = pd.DataFrame({
    "Partidas": partidas,
    "Wards/m": wpm,
    "Media wards/m": wpm_media(wpm,partidas)
}).fillna(0)
print(player_wpm_media)

                  Partidas  Wards/m  Media wards/m
playername                                        
 BLWeaksideWizar         2   0.7373           0.37
0909                     3   1.0198           0.34
0ri                     29   4.0025           0.14
1116                    74  27.0196           0.37
113                     67  16.5678           0.25
...                    ...      ...            ...
zeta                     5   6.0815           1.22
zorenous                 4   1.6814           0.42
zynts                   47  22.9196           0.49
zzeta                    5   1.5038           0.30
Évangelyne              21   6.0203           0.29

[2803 rows x 3 columns]


## Wards destruidos/min

In [103]:
# Obtengo la suma de guardianes destruidos por minuto
wdpm = (
    df_players.groupby("playername")["wcpm"]
      .sum()
)

In [104]:
# Función lambda para calcular la media de guardianes destruidos por minuto del jugador
wdpm_media = lambda wardsd, part: round(wardsd / part, 2)

In [105]:
player_wdpm_media = pd.DataFrame({
    "Partidas": partidas,
    "Wards_d/m": wdpm,
    "Media wards_d/m": wdpm_media(wdpm,partidas)
}).fillna(0)
print(player_wdpm_media)

                  Partidas  Wards_d/m  Media wards_d/m
playername                                            
 BLWeaksideWizar         2     0.0766             0.04
0909                     3     0.4524             0.15
0ri                     29     5.3642             0.18
1116                    74    12.7669             0.17
113                     67    18.5102             0.28
...                    ...        ...              ...
zeta                     5     1.4244             0.28
zorenous                 4     0.8133             0.20
zynts                   47     8.8874             0.19
zzeta                    5     1.1905             0.24
Évangelyne              21     3.9194             0.19

[2803 rows x 3 columns]


# ------------------------------------